In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("/Users/theerdhasjoy/BIA/ML/Dataset/student_performance_linear_regression.csv")

df.head()

,study_hours,attendance_percent,sleep_hours,previous_score,assignments_completed,internet_access,study_method,final_score
0,11.6,86.2,7.5,44.2,4,Yes,Self Study,87.2
1,8.7,71.1,4.7,36.3,5,No,Online,74.1
2,12.0,58.1,7.8,44.1,10,Yes,Online,77.4
3,3.4,66.6,5.1,75.9,9,Yes,Self Study,71.7
4,5.0,77.1,7.5,91.5,10,Yes,Online,86.5


In [3]:
# EDA

def eda(df):
    shape_df = df.shape
    info_df = df.info()
    desc_df = df.describe()
    missing_df = df.isnull().sum()
    unique_df = df.nunique()

    return shape_df, info_df, desc_df, missing_df, unique_df

eda_find = eda(df)

print(eda_find)


<class 'pandas.DataFrame'>
RangeIndex: 603 entries, 0 to 602
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   study_hours            591 non-null    float64
 1   attendance_percent     593 non-null    float64
 2   sleep_hours            603 non-null    float64
 3   previous_score         603 non-null    float64
 4   assignments_completed  603 non-null    int64  
 5   internet_access        603 non-null    str    
 6   study_method           595 non-null    str    
 7   final_score            603 non-null    float64
dtypes: float64(5), int64(1), str(2)
memory usage: 44.4 KB
((603, 8), None,        study_hours  attendance_percent  sleep_hours  previous_score  \
count   591.000000          593.000000   603.000000      603.000000   
mean      6.602707           78.580607     6.450580       64.175788   
std       3.425803           12.677990     1.469777       17.395857   
min       1.000000         

In [4]:
# Feature Engineering

num_cols = df.select_dtypes(include= np.number).columns.tolist()
cat_cols = df.select_dtypes(include= "object").columns.tolist()

print(f"Numerical columns are : {num_cols}")
print(f"Categorical columns are : {cat_cols}")

Numerical columns are : ['study_hours', 'attendance_percent', 'sleep_hours', 'previous_score', 'assignments_completed', 'final_score']
Categorical columns are : ['internet_access', 'study_method']


/var/folders/y5/lf_stmcj5nv2hnzy3q1b5pb80000gn/T/ipykernel_54832/2113439515.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include= "object").columns.tolist()


In [5]:
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mean())
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [6]:
df.isnull().sum()



study_hours              0
attendance_percent       0
sleep_hours              0
previous_score           0
assignments_completed    0
internet_access          0
study_method             0
final_score              0
dtype: int64

In [7]:
# Encoding

df["internet_access"] = df["internet_access"].map(
    {
        "No" : 0,
        "Yes" : 1
    }
)

In [8]:
# One-hot

df = pd.get_dummies(
    df,
    columns= ["study_method"],
    dtype= int
)

In [9]:
df.head()

,study_hours,attendance_percent,sleep_hours,previous_score,assignments_completed,internet_access,final_score,study_method_Coaching,study_method_Online,study_method_Self Study
0,11.6,86.2,7.5,44.2,4,1,87.2,0,0,1
1,8.7,71.1,4.7,36.3,5,0,74.1,0,1,0
2,12.0,58.1,7.8,44.1,10,1,77.4,0,1,0
3,3.4,66.6,5.1,75.9,9,1,71.7,0,0,1
4,5.0,77.1,7.5,91.5,10,1,86.5,0,1,0


In [10]:
# Train-test split

from sklearn.model_selection import train_test_split

X = df.drop(columns= "final_score")
y = df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state= 42
)




In [11]:
# Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [12]:
# Training

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
model = LinearRegression()

model.fit(X_train_scaled, y_train)
pred = model.predict(X_test_scaled)

print(f"MAE : {mean_absolute_error(y_test, pred)}")
print(f"MSE : {mean_squared_error(y_test, pred)}")
print(f"RMSE : {root_mean_squared_error(y_test, pred)}")
print(f"R2 : {r2_score(y_test, pred)}")



MAE : 4.378716165154298
MSE : 29.957301855511826
RMSE : 5.47332639767736
R2 : 0.7815941691992705


In [13]:
train_pred = model.predict(X_train_scaled)

print(f"Train R2 : {r2_score(y_train, train_pred)}")
print(f"Test R2 : {r2_score(y_test, pred)}")

Train R2 : 0.7629689190844003
Test R2 : 0.7815941691992705
